# Energy–Accuracy trade-off

This notebook combines the final energy measurements with per-sequence accuracy results and creates a publication-ready 2×2 figure:

- eH36M: Energy vs PCK@0.2 and Energy vs MPJPE
- DHP19: Energy vs PCK@0.2 and Energy vs MPJPE

Energy repetitions are first averaged **within each sequence**, so the statistical unit remains the sequence. Model-level points are then obtained by averaging across sequences.


In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------- CONFIGURATION -------------------------
ENERGY_CSV = '/data/MoveEnet_OFK_results/Energy/energy_accuracy_final_5hz_ofk/energy_measurements.csv'
ACCURACY_CSV = Path('accuracy_by_sample.csv')
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ACCURACY_CSV must contain:
# dataset,sample_id,model,pck,mpjpe

ENERGY_METRIC = 'measured_compute_energy_j'
ERROR_KIND = 'std'   # use 'sem' if you prefer standard error

MODEL_ORDER = {
    'h36m': ['movenet', 'moveenetofk', 'openpose', 'yolo'],
    'dhp19': ['movenet', 'moveenetofk', 'eventpointpose'],
}

MODEL_LABELS = {
    ('h36m', 'movenet'): 'MoveNet (50 Hz)',
    ('h36m', 'moveenetofk'): 'MoveEnetOFK (50 Hz net, 200 Hz flow)',
    ('h36m', 'openpose'): 'OpenPose (50 Hz)',
    ('h36m', 'yolo'): 'YOLOPose (50 Hz)',
    ('dhp19', 'movenet'): 'MoveNet (200 Hz)',
    ('dhp19', 'moveenetofk'): 'MoveEnetOFK (5 Hz net, 200 Hz flow)',
    ('dhp19', 'eventpointpose'): 'EventPointPose (200 Hz)',
}
DATASET_LABELS = {'h36m': 'eH36M', 'dhp19': 'DHP19'}


## Load and validate data

`energy_measurements.csv` contains one row per repetition. The accuracy file should contain one row per `(dataset, sample_id, model)`.


In [10]:
energy_raw = pd.read_csv(ENERGY_CSV)
accuracy = pd.read_csv(ACCURACY_CSV)

required_energy = {'dataset', 'sample_id', 'model', 'status', ENERGY_METRIC, 'wall_seconds'}
required_accuracy = {'dataset', 'sample_id', 'model', 'pck', 'mpjpe'}

missing_e = required_energy - set(energy_raw.columns)
missing_a = required_accuracy - set(accuracy.columns)
if missing_e:
    raise ValueError(f'Missing energy columns: {sorted(missing_e)}')
if missing_a:
    raise ValueError(f'Missing accuracy columns: {sorted(missing_a)}')

for df in (energy_raw, accuracy):
    df['dataset'] = df['dataset'].astype(str).str.lower()
    df['model'] = df['model'].astype(str).str.lower()

energy_ok = energy_raw[energy_raw['status'].astype(str).str.upper() == 'OK'].copy()
energy_ok[ENERGY_METRIC] = pd.to_numeric(energy_ok[ENERGY_METRIC], errors='coerce')
energy_ok['wall_seconds'] = pd.to_numeric(energy_ok['wall_seconds'], errors='coerce')
energy_ok = energy_ok.dropna(subset=[ENERGY_METRIC])

print('Successful energy rows:', len(energy_ok))
print('Accuracy rows:', len(accuracy))
display(energy_ok[['dataset', 'sample_id', 'model', ENERGY_METRIC, 'wall_seconds']].head())
display(accuracy.head())


FileNotFoundError: [Errno 2] No such file or directory: 'accuracy_by_sample.csv'

## Average repetitions within each sequence

Five repetitions of the same sequence are **not** treated as five independent samples.


In [ ]:
energy_by_sample = (
    energy_ok
    .groupby(['dataset', 'sample_id', 'model'], as_index=False)
    .agg(
        energy_j=(ENERGY_METRIC, 'mean'),
        energy_repeat_std_j=(ENERGY_METRIC, 'std'),
        wall_seconds=('wall_seconds', 'mean'),
        wall_repeat_std_s=('wall_seconds', 'std'),
        n_repetitions=(ENERGY_METRIC, 'size'),
    )
)

merged = energy_by_sample.merge(
    accuracy[['dataset', 'sample_id', 'model', 'pck', 'mpjpe']],
    on=['dataset', 'sample_id', 'model'],
    how='inner',
    validate='one_to_one',
)

print('Matched sequence-method pairs:', len(merged))
display(merged.head())

unmatched = energy_by_sample.merge(
    accuracy[['dataset', 'sample_id', 'model']],
    on=['dataset', 'sample_id', 'model'],
    how='left', indicator=True,
)
unmatched = unmatched[unmatched['_merge'] == 'left_only']
if len(unmatched):
    print('WARNING: energy results without matching accuracy:')
    display(unmatched[['dataset', 'sample_id', 'model']])


## Aggregate across sequences

The plotted point is the mean across sequences. Error bars describe variability **across sequences**, not variability between repeated energy measurements of the same sequence.


In [ ]:
def err(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if len(s) <= 1:
        return np.nan
    if ERROR_KIND == 'sem':
        return s.std(ddof=1) / np.sqrt(len(s))
    return s.std(ddof=1)

summary = (
    merged
    .groupby(['dataset', 'model'], as_index=False)
    .agg(
        energy_mean_j=('energy_j', 'mean'),
        energy_err_j=('energy_j', err),
        pck_mean=('pck', 'mean'),
        pck_err=('pck', err),
        mpjpe_mean=('mpjpe', 'mean'),
        mpjpe_err=('mpjpe', err),
        wall_mean_s=('wall_seconds', 'mean'),
        n_sequences=('sample_id', 'nunique'),
    )
)
display(summary.sort_values(['dataset', 'model']))


## Final figure

- PCK: upper-left is better (higher accuracy, lower energy).
- MPJPE: lower-left is better (lower error, lower energy).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

for row, dataset in enumerate(['h36m', 'dhp19']):
    ds = summary[summary['dataset'] == dataset]
    for col, metric in enumerate(['pck', 'mpjpe']):
        ax = axes[row, col]
        for model in MODEL_ORDER[dataset]:
            r = ds[ds['model'] == model]
            if r.empty:
                continue
            r = r.iloc[0]
            y_mean = r[f'{metric}_mean']
            y_err = r[f'{metric}_err']
            label = MODEL_LABELS.get((dataset, model), model)

            ax.errorbar(
                r['energy_mean_j'], y_mean,
                xerr=r['energy_err_j'], yerr=y_err,
                fmt='o', capsize=3, markersize=7,
            )
            ax.annotate(label, (r['energy_mean_j'], y_mean),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)

        ax.set_xlabel('Measured compute energy [J / sequence]')
        if metric == 'pck':
            ax.set_ylabel('PCK@0.2')
            ax.set_title(f"{DATASET_LABELS[dataset]} — Energy vs PCK@0.2")
        else:
            ax.set_ylabel('MPJPE')
            ax.set_title(f"{DATASET_LABELS[dataset]} — Energy vs MPJPE")
        ax.grid(alpha=0.25)

fig.suptitle('Energy–Accuracy Trade-off', fontsize=14)
png_path = OUTPUT_DIR / 'energy_accuracy_tradeoff.png'
pdf_path = OUTPUT_DIR / 'energy_accuracy_tradeoff.pdf'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
plt.show()
print('Saved:', png_path)
print('Saved:', pdf_path)


In [ ]:
summary_path = OUTPUT_DIR / 'energy_accuracy_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
